## 欢迎进入 Notebook  

这里你可以编写代码，文档  

### 关于文件目录  


**project**：project 目录是本项目的工作空间，可以把将项目运行有关的所有文件放在这里，目录中文件的增、删、改操作都会被保留  


**input**：input 目录是数据集的挂载位置，所有挂载进项目的数据集都在这里，未挂载数据集时 input 目录被隐藏  


**temp**：temp 目录是临时磁盘空间，训练或分析过程中产生的不必要文件可以存放在这里，目录中的文件不会保存  


In [4]:
import numpy as np
import pandas as pd
import time

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

########################################
# 0. 基础参数
########################################
csv_path    = "/home/mw/input/abc6966/副本ruc_Class25Q2_train_rent91.csv"  # ← 你那份数据
target_col  = "Price"    # 这里是租金/价格列
K_CLUSTERS  = 80
TEST_RATIO  = 0.2
RANDOM_SEED = 111
CV_FOLDS    = 6

########################################
# 1. 读数据
########################################
df = pd.read_csv(csv_path)
print("读入数据:", df.shape)

########################################
# 2. 转成数值型 & 填缺失
########################################
num_cols_candidate = [
    "Price",
    "lon","lat",
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "租赁方式_整租",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器","设施_冰箱",
    "设施_天然气","设施_电视","设施_暖气","设施_宽带"
]

# 只保留当前真正在 df 里的列
num_cols = [c for c in num_cols_candidate if c in df.columns]

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 对电梯列特别注意，如果缺失就当0
if "电梯_有" in df.columns:
    df["电梯_有"] = df["电梯_有"].fillna(0)

########################################
# 3. 用每条房源来聚类成80个“市场片区” (market_cluster80)
#    使用位置+品质+均价信息
########################################
cluster_basis_cols = []
for c in [
    "lon","lat",
    "房龄","绿化率","容积率","物业费",
    "房屋总数","楼栋总数",
    "Price"
]:
    if c in df.columns:
        cluster_basis_cols.append(c)

if len(cluster_basis_cols) < 2:
    raise ValueError("聚类特征太少，至少需要 lon/lat 之类的位置信息。请检查列名。")

cluster_data = df[cluster_basis_cols].copy()

# 数值化 + 缺失值用列中位数填
for c in cluster_data.columns:
    cluster_data[c] = pd.to_numeric(cluster_data[c], errors="coerce")
cluster_data = cluster_data.fillna(cluster_data.median(numeric_only=True))

# 标准化，再 KMeans
scaler_cluster = StandardScaler()
cluster_scaled = scaler_cluster.fit_transform(cluster_data)

K = min(K_CLUSTERS, len(df))
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["market_cluster80"] = kmeans.fit_predict(cluster_scaled).astype(int)

########################################
# 4. 造交互项（交叉项）
########################################
# 面积 × 房龄
if ("面积" in df.columns) and ("房龄" in df.columns):
    df["交互_面积x房龄"] = pd.to_numeric(df["面积"], errors="coerce") * \
                         pd.to_numeric(df["房龄"], errors="coerce")

# 面积 × 楼层_高层
if ("面积" in df.columns) and ("楼层_高层" in df.columns):
    df["交互_面积x高层"] = pd.to_numeric(df["面积"], errors="coerce") * \
                        pd.to_numeric(df["楼层_高层"], errors="coerce")

# 楼层_高层 × 电梯_有
if ("楼层_高层" in df.columns) and ("电梯_有" in df.columns):
    df["交互_高层x电梯有"] = pd.to_numeric(df["楼层_高层"], errors="coerce") * \
                          pd.to_numeric(df["电梯_有"], errors="coerce")

########################################
# 5. 组最终特征 X
########################################
base_feature_cols = [
    # 物理/结构
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费","房龄",
    # 区位
    "lon","lat","城市",
    # 楼层、电梯
    "楼层_中层","楼层_高层","楼层_地下室","电梯_有",
    # 付款方式 dummy
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    # 租赁方式
    "租赁方式_整租",
    # 朝向 dummy
    "朝向_东","朝向_南","朝向_西","朝向_北",
    # 时间 dummy
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    # 配套设施 dummy
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带",
    # 交互项
    "交互_面积x房龄","交互_面积x高层","交互_高层x电梯有",
]

# 有的才留
base_feature_cols = [c for c in base_feature_cols if c in df.columns]

# 先把 '城市' 也dummy化，因为你现在没有 city_00/city_01...，但有城市列
if "城市" in df.columns:
    city_dum = pd.get_dummies(df["城市"], prefix="city", drop_first=False).astype(int)
else:
    city_dum = pd.DataFrame(index=df.index)

# 再把 market_cluster80 dummy 化
cluster_dum = pd.get_dummies(
    df["market_cluster80"],
    prefix="cluster80",
    drop_first=False
).astype(int)

# 数值特征本体
X_base = df[base_feature_cols].copy()

# 把所有列都转成数值，缺失填中位数
for c in X_base.columns:
    X_base[c] = pd.to_numeric(X_base[c], errors="coerce")
X_base = X_base.fillna(X_base.median(numeric_only=True))

# 拼成最终 X_full
X_full = pd.concat(
    [X_base.reset_index(drop=True),
     city_dum.reset_index(drop=True),
     cluster_dum.reset_index(drop=True)],
    axis=1
)

# 丢掉标准差=0的列（常数列）
X_full = X_full.loc[:, X_full.std(axis=0) > 0]

# 目标 y
y_all = pd.to_numeric(df[target_col], errors="coerce")
y_all = y_all.fillna(y_all.median())

########################################
# 6. 用 IQR 去极端值
########################################
Q1 = y_all.quantile(0.25)
Q3 = y_all.quantile(0.75)
IQR = Q3 - Q1
lower_cut = Q1 - 1.5 * IQR
upper_cut = Q3 + 1.5 * IQR
mask_ok = (y_all >= lower_cut) & (y_all <= upper_cut)

X_ok = X_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_all.loc[mask_ok].reset_index(drop=True)

print("用于回归的样本量:", X_ok.shape[0], "特征数:", X_ok.shape[1])

########################################
# 7. 划分训练/测试
########################################
X_train, X_test, y_train, y_test = train_test_split(
    X_ok, y_ok,
    test_size=TEST_RATIO,
    random_state=RANDOM_SEED
)

########################################
# 8. 建立线性模型 (OLS) + 标准化
########################################
ols_pipe = make_pipeline(
    StandardScaler(with_mean=False),  # 保留稀疏/哑变量结构
    LinearRegression()
)

t0 = time.time()
ols_pipe.fit(X_train, y_train)
t1 = time.time()
print(f"OLS 拟合完成，用时 {t1 - t0:.2f} 秒")

########################################
# 9. 训练集 / 测试集 指标
########################################
# 训练集
y_pred_tr = ols_pipe.predict(X_train)
train_MAE  = mean_absolute_error(y_train, y_pred_tr)
train_MSE  = mean_squared_error(y_train, y_pred_tr)
train_RMSE = np.sqrt(train_MSE)
train_R2   = r2_score(y_train, y_pred_tr)

# 测试集
y_pred_te = ols_pipe.predict(X_test)
test_MAE  = mean_absolute_error(y_test, y_pred_te)
test_MSE  = mean_squared_error(y_test, y_pred_te)
test_RMSE = np.sqrt(test_MSE)
test_R2   = r2_score(y_test, y_pred_te)

print("\n========== OLS (K=80聚类 + 城市 + 时间 + 交互项) ==========")
print("[训练集 / in-sample]")
print(f"MAE  : {train_MAE:.4f}")
print(f"MSE  : {train_MSE:.4f}")
print(f"RMSE : {train_RMSE:.4f}")
print(f"R^2  : {train_R2:.4f}")
print("")
print("[测试集 / out-of-sample]")
print(f"MAE  : {test_MAE:.4f}")
print(f"MSE  : {test_MSE:.4f}")
print(f"RMSE : {test_RMSE:.4f}")
print(f"R^2  : {test_R2:.4f}")
print("==========================================================")

########################################
# 10. 6折交叉验证 (全量去极值后的X_ok, y_ok)
########################################
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

cv_mae_list  = []
cv_mse_list  = []
cv_rmse_list = []
cv_r2_list   = []

fold_id = 0
for tr_idx, va_idx in kf.split(X_ok):
    fold_id += 1
    X_tr = X_ok.iloc[tr_idx]
    X_va = X_ok.iloc[va_idx]
    y_tr = y_ok.iloc[tr_idx]
    y_va = y_ok.iloc[va_idx]

    pipe_cv = make_pipeline(
        StandardScaler(with_mean=False),
        LinearRegression()
    )
    pipe_cv.fit(X_tr, y_tr)
    y_hat = pipe_cv.predict(X_va)

    mae_val  = mean_absolute_error(y_va, y_hat)
    mse_val  = mean_squared_error(y_va, y_hat)
    rmse_val = np.sqrt(mse_val)
    r2_val   = r2_score(y_va, y_hat)

    cv_mae_list.append(mae_val)
    cv_mse_list.append(mse_val)
    cv_rmse_list.append(rmse_val)
    cv_r2_list.append(r2_val)

    print(f"[CV折 {fold_id}/{CV_FOLDS}] "
          f"MAE={mae_val:.4f} | MSE={mse_val:.4f} | RMSE={rmse_val:.4f} | R^2={r2_val:.4f}")

cv_MAE_mean   = float(np.mean(cv_mae_list))
cv_MAE_std    = float(np.std(cv_mae_list))
cv_MSE_mean   = float(np.mean(cv_mse_list))
cv_MSE_std    = float(np.std(cv_mse_list))
cv_RMSE_mean  = float(np.mean(cv_rmse_list))
cv_RMSE_std   = float(np.std(cv_rmse_list))
cv_R2_mean    = float(np.mean(cv_r2_list))
cv_R2_std     = float(np.std(cv_r2_list))

print("\n[6折交叉验证 - 汇总平均±std]")
print(f"CV MAE   : {cv_MAE_mean:.4f} (std {cv_MAE_std:.4f})")
print(f"CV MSE   : {cv_MSE_mean:.4f} (std {cv_MSE_std:.4f})")
print(f"CV RMSE  : {cv_RMSE_mean:.4f} (std {cv_RMSE_std:.4f})")
print(f"CV R^2   : {cv_R2_mean:.4f} (std {cv_R2_std:.4f})")
print("==========================================================")

########################################
# 11. 导出预测 + cluster80
########################################
pred_all = ols_pipe.predict(X_ok)

export_df = pd.concat(
    [
        df.loc[mask_ok].reset_index(drop=True),
        pd.Series(pred_all, name="OLS_predicted_price_cluster80_interact")
    ],
    axis=1
)
out_path = "rent_with_cluster80_interact.csv"
export_df.to_csv(out_path, index=False)
print("已导出 ->", out_path)


读入数据: (98899, 47)
用于回归的样本量: 93365 特征数: 141
OLS 拟合完成，用时 3.23 秒

========== OLS (K=80聚类 + 城市 + 时间 + 交互项) ==========
[训练集 / in-sample]
MAE  : 108467.7875
MSE  : 22483411177.4644
RMSE : 149944.6937
R^2  : 0.7548

[测试集 / out-of-sample]
MAE  : 108784.5758
MSE  : 22480385620.0674
RMSE : 149934.6045
R^2  : 0.7578
[CV折 1/6] MAE=108890.3992 | MSE=22600814799.0842 | RMSE=150335.6737 | R^2=0.7578
[CV折 2/6] MAE=108792.4948 | MSE=22534092105.3714 | RMSE=150113.5973 | R^2=0.7561
[CV折 3/6] MAE=108269.7324 | MSE=22483398591.2237 | RMSE=149944.6518 | R^2=0.7556
[CV折 4/6] MAE=108389.5979 | MSE=22443439353.6926 | RMSE=149811.3459 | R^2=0.7555
[CV折 5/6] MAE=108538.7667 | MSE=22448363831.4925 | RMSE=149827.7806 | R^2=0.7550
[CV折 6/6] MAE=109290.7813 | MSE=22806092255.4903 | RMSE=151016.8608 | R^2=0.7476

[6折交叉验证 - 汇总平均±std]
CV MAE   : 108695.2954 (std 341.8510)
CV MSE   : 22552700156.0591 (std 125496037.7359)
CV RMSE  : 150174.9850 (std 417.1694)
CV R^2   : 0.7546 (std 0.0033)
已导出 -> rent_with_cluster80_int

In [6]:
import numpy as np
import pandas as pd
import time

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.linear_model import Lasso
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

########################################
# 0. 配置参数
########################################
csv_path       = "//home/mw/input/abc6966/副本ruc_Class25Q2_train_rent91.csv"
target_col     = "Price"

K_CLUSTERS     = 80          # 聚类成80个市场片区
SUBSET_SIZE    = 2000        # 调参只用这么多行（如果总样本更少，就会自动用全量）
CV_FOLDS       = 6           # 6折交叉验证
RANDOM_SEED    = 42
ALPHAS_TO_TRY  = [1.0, 5.0, 10.0, 50.0, 100.0, 500.0, 1000.0, 5000.0, 10000.0]

print("【Lasso小样本调参】开始", flush=True)

########################################
# 1. 读数据
########################################
df = pd.read_csv(csv_path)
print("数据维度:", df.shape, flush=True)

########################################
# 2. 数值列 -> numeric
########################################
num_cols_candidate = [
    "Price",
    "lon","lat",
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "租赁方式_整租",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带"
]
num_cols = [c for c in num_cols_candidate if c in df.columns]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if "电梯_有" in df.columns:
    df["电梯_有"] = df["电梯_有"].fillna(0)

########################################
# 3. 创建 market_cluster80 via KMeans
#    用位置信息+楼盘质量指标+房价水平
########################################
cluster_basis_cols = []
for c in [
    "lon","lat",
    "房龄","绿化率","容积率","物业费",
    "房屋总数","楼栋总数",
    "Price"
]:
    if c in df.columns:
        cluster_basis_cols.append(c)

if len(cluster_basis_cols) < 2:
    raise ValueError("聚类特征太少，至少要 lon/lat 之类的位置列。")

cluster_data = df[cluster_basis_cols].copy()
for c in cluster_data.columns:
    cluster_data[c] = pd.to_numeric(cluster_data[c], errors="coerce")
cluster_data = cluster_data.fillna(cluster_data.median(numeric_only=True))

scaler_cluster = StandardScaler()
cluster_scaled = scaler_cluster.fit_transform(cluster_data)

K = min(K_CLUSTERS, len(df))
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["market_cluster80"] = kmeans.fit_predict(cluster_scaled).astype(int)

########################################
# 4. 加交互项（线性模型里允许不同斜率）
########################################
# 面积×房龄
if ("面积" in df.columns) and ("房龄" in df.columns):
    df["交互_面积x房龄"] = pd.to_numeric(df["面积"], errors="coerce") * \
                         pd.to_numeric(df["房龄"], errors="coerce")

# 面积×楼层_高层
if ("面积" in df.columns) and ("楼层_高层" in df.columns):
    df["交互_面积x高层"] = pd.to_numeric(df["面积"], errors="coerce") * \
                        pd.to_numeric(df["楼层_高层"], errors="coerce")

# 楼层_高层×电梯_有
if ("楼层_高层" in df.columns) and ("电梯_有" in df.columns):
    df["交互_高层x电梯有"] = pd.to_numeric(df["楼层_高层"], errors="coerce") * \
                          pd.to_numeric(df["电梯_有"], errors="coerce")

########################################
# 5. 构造特征矩阵 X_full
########################################
# 先把城市 dummy 化（城市是分类）
if "城市" in df.columns:
    city_dum = pd.get_dummies(df["城市"], prefix="city", drop_first=False).astype(int)
else:
    city_dum = pd.DataFrame(index=df.index)

# market_cluster80 dummy
cluster_dum = pd.get_dummies(
    df["market_cluster80"],
    prefix="cluster80",
    drop_first=False
).astype(int)

# 基础线性特征 + 时间 + 付款方式 + 设施 + 交互
base_feature_cols = [
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "lon","lat",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "租赁方式_整租",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带",
    "交互_面积x房龄","交互_面积x高层","交互_高层x电梯有",
]
base_feature_cols = [c for c in base_feature_cols if c in df.columns]

X_base = df[base_feature_cols].copy()
for c in X_base.columns:
    X_base[c] = pd.to_numeric(X_base[c], errors="coerce")
X_base = X_base.fillna(X_base.median(numeric_only=True))

# 拼接最终特征
X_full = pd.concat(
    [X_base.reset_index(drop=True),
     city_dum.reset_index(drop=True),
     cluster_dum.reset_index(drop=True)],
    axis=1
)

# 去掉常数列（std=0）
X_full = X_full.loc[:, X_full.std(axis=0) > 0]

# y
y_all = pd.to_numeric(df[target_col], errors="coerce")
y_all = y_all.fillna(y_all.median())

########################################
# 6. IQR 去极值
########################################
Q1 = y_all.quantile(0.25)
Q3 = y_all.quantile(0.75)
IQR = Q3 - Q1
lower_cut = Q1 - 1.5 * IQR
upper_cut = Q3 + 1.5 * IQR

mask_ok = (y_all >= lower_cut) & (y_all <= upper_cut)

X_ok = X_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_all.loc[mask_ok].reset_index(drop=True)

print("用于Lasso调参的可用样本量:", X_ok.shape[0], "特征数:", X_ok.shape[1], flush=True)

########################################
# 7. 抽一个子样本来调参
########################################
n_total = len(X_ok)
n_take  = min(SUBSET_SIZE, n_total)

rng = np.random.RandomState(RANDOM_SEED)
idx = rng.choice(n_total, size=n_take, replace=False)

X_sub = X_ok.iloc[idx].copy()
y_sub = y_ok.iloc[idx].copy()

print(f"抽到子样本: {len(X_sub)} / {n_total} 行", flush=True)

########################################
# 8. 6折CV，扫描 ALPHAS_TO_TRY
########################################
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=111)

alpha_results = []

for a in ALPHAS_TO_TRY:
    print(f"\n=== alpha = {a} 开始 ===", flush=True)
    t_alpha = time.time()

    fold_mae_list  = []
    fold_rmse_list = []
    fold_r2_list   = []

    fold_id = 0
    for tr_idx, va_idx in kf.split(X_sub):
        fold_id += 1
        X_tr = X_sub.iloc[tr_idx]
        X_va = X_sub.iloc[va_idx]
        y_tr = y_sub.iloc[tr_idx]
        y_va = y_sub.iloc[va_idx]

        pipe = make_pipeline(
            StandardScaler(with_mean=False),
            Lasso(
                alpha=a,
                fit_intercept=True,
                max_iter=50000,
                random_state=42
            )
        )

        t_fold = time.time()
        pipe.fit(X_tr, y_tr)
        y_hat = pipe.predict(X_va)

        mae_val  = mean_absolute_error(y_va, y_hat)
        mse_val  = mean_squared_error(y_va, y_hat)
        rmse_val = np.sqrt(mse_val)
        r2_val   = r2_score(y_va, y_hat)

        fold_mae_list.append(mae_val)
        fold_rmse_list.append(rmse_val)
        fold_r2_list.append(r2_val)

        print(f"  折 {fold_id}/{CV_FOLDS} | "
              f"MAE={mae_val:,.4f} | RMSE={rmse_val:,.4f} | R²={r2_val:.4f} "
              f"| 用时 {time.time()-t_fold:.2f}s", flush=True)

    mae_mean   = float(np.mean(fold_mae_list))
    mae_std    = float(np.std(fold_mae_list))
    rmse_mean  = float(np.mean(fold_rmse_list))
    rmse_std   = float(np.std(fold_rmse_list))
    r2_mean    = float(np.mean(fold_r2_list))
    r2_std     = float(np.std(fold_r2_list))
    elapsed_a  = time.time() - t_alpha

    alpha_results.append({
        "alpha": a,
        "cv_mae_mean": mae_mean,
        "cv_mae_std":  mae_std,
        "cv_rmse_mean": rmse_mean,
        "cv_rmse_std":  rmse_std,
        "cv_r2_mean":  r2_mean,
        "cv_r2_std":   r2_std,
        "time_sec": elapsed_a
    })

    print(f"=== alpha = {a} 结束 ===", flush=True)
    print(f"    -> CV MAE均值={mae_mean:,.4f} ± {mae_std:,.4f}", flush=True)
    print(f"    -> CV RMSE均值={rmse_mean:,.4f} ± {rmse_std:,.4f}", flush=True)
    print(f"    -> CV R²均值={r2_mean:.4f} ± {r2_std:.4f}", flush=True)
    print(f"    -> 耗时 {elapsed_a:.2f}s", flush=True)

########################################
# 9. 汇总所有alpha，选最优
########################################
results_df = pd.DataFrame(alpha_results)
results_sorted = results_df.sort_values("cv_mae_mean").reset_index(drop=True)

best_row = results_sorted.iloc[0]
best_alpha = best_row["alpha"]

print("\n========== Lasso小样本调参结果 (按 MAE 从小到大) ==========")
print(results_sorted.to_string(index=False))
print("========================================================")
print(f"\n>>> 结论：当前数据下，最优 alpha = {best_alpha}\n", flush=True)


【Lasso小样本调参】开始
数据维度: (98899, 47)
用于Lasso调参的可用样本量: 93365 特征数: 140
抽到子样本: 2000 / 93365 行

=== alpha = 1.0 开始 ===
  折 1/6 | MAE=117,004.6918 | RMSE=164,994.3029 | R²=0.7202 | 用时 216.98s
  折 2/6 | MAE=115,707.3141 | RMSE=156,263.2864 | R²=0.7548 | 用时 241.22s


/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:634: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.215e+10, tolerance: 1.514e+10
  model = cd_fast.enet_coordinate_descent(


  折 3/6 | MAE=108,782.1386 | RMSE=146,422.0981 | R²=0.7555 | 用时 213.79s
  折 4/6 | MAE=121,930.0905 | RMSE=164,986.6385 | R²=0.6850 | 用时 113.10s
  折 5/6 | MAE=103,353.1693 | RMSE=149,025.4717 | R²=0.7618 | 用时 246.00s


/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:634: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.828e+11, tolerance: 1.538e+10
  model = cd_fast.enet_coordinate_descent(


  折 6/6 | MAE=113,273.0332 | RMSE=155,704.1146 | R²=0.7271 | 用时 240.80s
=== alpha = 1.0 结束 ===
    -> CV MAE均值=113,341.7396 ± 5,962.9293
    -> CV RMSE均值=156,232.6520 ± 7,093.0660
    -> CV R²均值=0.7341 ± 0.0268
    -> 耗时 1272.68s

=== alpha = 5.0 开始 ===
  折 1/6 | MAE=116,989.3406 | RMSE=164,998.2731 | R²=0.7202 | 用时 106.40s
  折 2/6 | MAE=115,653.9299 | RMSE=156,227.1497 | R²=0.7549 | 用时 89.89s
  折 3/6 | MAE=108,734.6081 | RMSE=146,391.5130 | R²=0.7556 | 用时 89.39s
  折 4/6 | MAE=121,911.2192 | RMSE=164,900.7733 | R²=0.6853 | 用时 59.19s
  折 5/6 | MAE=103,313.9970 | RMSE=149,028.3194 | R²=0.7617 | 用时 240.00s


/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:634: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.124e+11, tolerance: 1.538e+10
  model = cd_fast.enet_coordinate_descent(


  折 6/6 | MAE=113,236.2388 | RMSE=155,618.8382 | R²=0.7274 | 用时 92.10s
=== alpha = 5.0 结束 ===
    -> CV MAE均值=113,306.5556 ± 5,970.3738
    -> CV RMSE均值=156,194.1445 ± 7,083.9110
    -> CV R²均值=0.7342 ± 0.0267
    -> 耗时 677.50s

=== alpha = 10.0 开始 ===
  折 1/6 | MAE=116,977.5674 | RMSE=165,027.1714 | R²=0.7201 | 用时 72.58s
  折 2/6 | MAE=115,587.5097 | RMSE=156,186.0753 | R²=0.7551 | 用时 84.19s
  折 3/6 | MAE=108,678.3302 | RMSE=146,357.1760 | R²=0.7558 | 用时 81.90s
  折 4/6 | MAE=121,864.1978 | RMSE=164,852.9586 | R²=0.6855 | 用时 52.40s
  折 5/6 | MAE=103,264.5874 | RMSE=149,033.3645 | R²=0.7617 | 用时 255.10s


/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:634: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.152e+11, tolerance: 1.538e+10
  model = cd_fast.enet_coordinate_descent(


  折 6/6 | MAE=113,208.9318 | RMSE=155,608.0644 | R²=0.7274 | 用时 85.11s
=== alpha = 10.0 结束 ===
    -> CV MAE均值=113,263.5207 ± 5,974.5618
    -> CV RMSE均值=156,177.4684 ± 7,087.3365
    -> CV R²均值=0.7342 ± 0.0267
    -> 耗时 631.90s

=== alpha = 50.0 开始 ===
  折 1/6 | MAE=117,130.5522 | RMSE=165,347.0810 | R²=0.7190 | 用时 44.60s
  折 2/6 | MAE=115,092.0800 | RMSE=155,975.0129 | R²=0.7557 | 用时 35.29s
  折 3/6 | MAE=108,406.8901 | RMSE=146,243.7831 | R²=0.7561 | 用时 36.10s
  折 4/6 | MAE=121,782.9035 | RMSE=164,450.5392 | R²=0.6870 | 用时 51.81s
  折 5/6 | MAE=102,937.1337 | RMSE=149,135.4217 | R²=0.7614 | 用时 60.20s
  折 6/6 | MAE=113,176.9742 | RMSE=155,807.5717 | R²=0.7267 | 用时 36.09s
=== alpha = 50.0 结束 ===
    -> CV MAE均值=113,087.7556 ± 6,067.9583
    -> CV RMSE均值=156,159.9016 ± 7,082.4899
    -> CV R²均值=0.7343 ± 0.0264
    -> 耗时 264.60s

=== alpha = 100.0 开始 ===
  折 1/6 | MAE=117,561.8847 | RMSE=166,105.4324 | R²=0.7164 | 用时 23.20s
  折 2/6 | MAE=114,588.5813 | RMSE=155,946.2298 | R²=0.7558 | 用时 2

In [7]:
import numpy as np
import pandas as pd
import time

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

########################################
# 配置
########################################
csv_path       = "/home/mw/input/abc6966/副本ruc_Class25Q2_train_rent91.csv"
target_col     = "Price"
K_CLUSTERS     = 80
TEST_RATIO     = 0.2
RANDOM_SEED    = 111
KF_SPLITS      = 6
BEST_ALPHA     = 1000.0   # ← 你刚跑出来的最优alpha

print("【Lasso最终版全量评估】alpha =", BEST_ALPHA, flush=True)

########################################
# 1. 读数据
########################################
df = pd.read_csv(csv_path)
print("数据维度:", df.shape, flush=True)

########################################
# 2. 转数值并填缺失
########################################
num_cols_candidate = [
    "Price",
    "lon","lat",
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "租赁方式_整租",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带"
]
num_cols = [c for c in num_cols_candidate if c in df.columns]

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 电梯_有 缺失就当0（没写=没电梯）
if "电梯_有" in df.columns:
    df["电梯_有"] = df["电梯_有"].fillna(0)

########################################
# 3. KMeans 聚类成 80 个市场片区 (直接对每套房聚，不用"板块")
########################################
cluster_basis_cols = []
for c in [
    "lon","lat",
    "房龄","绿化率","容积率","物业费",
    "房屋总数","楼栋总数",
    "Price"
]:
    if c in df.columns:
        cluster_basis_cols.append(c)

cluster_data = df[cluster_basis_cols].copy()
for c in cluster_data.columns:
    cluster_data[c] = pd.to_numeric(cluster_data[c], errors="coerce")
cluster_data = cluster_data.fillna(cluster_data.median(numeric_only=True))

scaler_cluster = StandardScaler()
cluster_scaled = scaler_cluster.fit_transform(cluster_data)

K = min(K_CLUSTERS, len(df))
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["market_cluster80"] = kmeans.fit_predict(cluster_scaled).astype(int)

########################################
# 4. 交互项（线性模型允许斜率随条件变）
########################################
if ("面积" in df.columns) and ("房龄" in df.columns):
    df["交互_面积x房龄"] = pd.to_numeric(df["面积"], errors="coerce") * \
                         pd.to_numeric(df["房龄"], errors="coerce")

if ("面积" in df.columns) and ("楼层_高层" in df.columns):
    df["交互_面积x高层"] = pd.to_numeric(df["面积"], errors="coerce") * \
                        pd.to_numeric(df["楼层_高层"], errors="coerce")

if ("楼层_高层" in df.columns) and ("电梯_有" in df.columns):
    df["交互_高层x电梯有"] = pd.to_numeric(df["楼层_高层"], errors="coerce") * \
                          pd.to_numeric(df["电梯_有"], errors="coerce")

########################################
# 5. 构建特征矩阵 X_full
########################################
# 城市 dummy
if "城市" in df.columns:
    city_dum = pd.get_dummies(df["城市"], prefix="city", drop_first=False).astype(int)
else:
    city_dum = pd.DataFrame(index=df.index)

# market_cluster80 dummy
cluster_dum = pd.get_dummies(
    df["market_cluster80"],
    prefix="cluster80",
    drop_first=False
).astype(int)

# 主体特征列（结构、位置、楼层、电梯、配套、时间、付款方式、租赁方式、交互项）
base_feature_cols = [
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "lon","lat",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "租赁方式_整租",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带",
    "交互_面积x房龄","交互_面积x高层","交互_高层x电梯有",
]
base_feature_cols = [c for c in base_feature_cols if c in df.columns]

X_base = df[base_feature_cols].copy()
for c in X_base.columns:
    X_base[c] = pd.to_numeric(X_base[c], errors="coerce")
X_base = X_base.fillna(X_base.median(numeric_only=True))

X_full = pd.concat(
    [X_base.reset_index(drop=True),
     city_dum.reset_index(drop=True),
     cluster_dum.reset_index(drop=True)],
    axis=1
)

# 去掉标准差为0的列（常数列）
X_full = X_full.loc[:, X_full.std(axis=0) > 0]

# y 向量
y_all = pd.to_numeric(df[target_col], errors="coerce")
y_all = y_all.fillna(y_all.median())

########################################
# 6. IQR 去极端值
########################################
Q1 = y_all.quantile(0.25)
Q3 = y_all.quantile(0.75)
IQR = Q3 - Q1
lower_cut = Q1 - 1.5 * IQR
upper_cut = Q3 + 1.5 * IQR
mask_ok = (y_all >= lower_cut) & (y_all <= upper_cut)

X_ok = X_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_all.loc[mask_ok].reset_index(drop=True)

print("最终可用样本量:", X_ok.shape[0], "特征数:", X_ok.shape[1], flush=True)

########################################
# 7. 切训练 / 测试
########################################
X_train, X_test, y_train, y_test = train_test_split(
    X_ok, y_ok,
    test_size=TEST_RATIO,
    random_state=RANDOM_SEED
)

print("训练集:", X_train.shape, y_train.shape, flush=True)
print("测试集:",  X_test.shape,  y_test.shape,  flush=True)

########################################
# 8. 定义最终 Lasso 模型 (用 BEST_ALPHA)
########################################
lasso_pipe = make_pipeline(
    StandardScaler(with_mean=False),  # 稀疏/哑变量友好
    Lasso(
        alpha=BEST_ALPHA,
        fit_intercept=True,
        max_iter=10000,    # 这已经够了，10000比50000快很多
        random_state=42
    )
)

########################################
# 9. 拟合训练集
########################################
print("\n[步骤1] 拟合训练集...", flush=True)
t0 = time.time()
lasso_pipe.fit(X_train, y_train)
print(f"拟合完成，用时 {time.time()-t0:.2f} 秒", flush=True)

########################################
# 10. 训练集 / 测试集 指标
########################################
print("\n[步骤2] 训练集/测试集指标...", flush=True)

# 训练集
y_pred_tr = lasso_pipe.predict(X_train)
train_MAE  = mean_absolute_error(y_train, y_pred_tr)
train_MSE  = mean_squared_error(y_train, y_pred_tr)
train_RMSE = np.sqrt(train_MSE)
train_R2   = r2_score(y_train, y_pred_tr)

# 测试集
y_pred_te = lasso_pipe.predict(X_test)
test_MAE  = mean_absolute_error(y_test, y_pred_te)
test_MSE  = mean_squared_error(y_test, y_pred_te)
test_RMSE = np.sqrt(test_MSE)
test_R2   = r2_score(y_test, y_pred_te)

print("【训练集 / in-sample】", flush=True)
print(f" MAE  : {train_MAE:,.4f}", flush=True)
print(f" MSE  : {train_MSE:,.4f}", flush=True)
print(f" RMSE : {train_RMSE:,.4f}", flush=True)
print(f" R^2  : {train_R2:.4f}", flush=True)

print("\n【测试集 / out-of-sample】", flush=True)
print(f" MAE  : {test_MAE:,.4f}", flush=True)
print(f" MSE  : {test_MSE:,.4f}", flush=True)
print(f" RMSE : {test_RMSE:,.4f}", flush=True)
print(f" R^2  : {test_R2:.4f}", flush=True)

########################################
# 11. 全量 6 折交叉验证 (X_ok, y_ok)
########################################
print("\n[步骤3] 6折交叉验证 (全量X_ok,y_ok)...", flush=True)

kf = KFold(n_splits=KF_SPLITS, shuffle=True, random_state=RANDOM_SEED)

cv_mae_list  = []
cv_mse_list  = []
cv_rmse_list = []
cv_r2_list   = []

fold_id = 0
for tr_idx, va_idx in kf.split(X_ok):
    fold_id += 1
    X_tr = X_ok.iloc[tr_idx]
    X_va = X_ok.iloc[va_idx]
    y_tr = y_ok.iloc[tr_idx]
    y_va = y_ok.iloc[va_idx]

    pipe_cv = make_pipeline(
        StandardScaler(with_mean=False),
        Lasso(
            alpha=BEST_ALPHA,
            fit_intercept=True,
            max_iter=10000,
            random_state=42
        )
    )

    t_fold = time.time()
    pipe_cv.fit(X_tr, y_tr)
    y_hat = pipe_cv.predict(X_va)

    mae_val  = mean_absolute_error(y_va, y_hat)
    mse_val  = mean_squared_error(y_va, y_hat)
    rmse_val = np.sqrt(mse_val)
    r2_val   = r2_score(y_va, y_hat)

    cv_mae_list.append(mae_val)
    cv_mse_list.append(mse_val)
    cv_rmse_list.append(rmse_val)
    cv_r2_list.append(r2_val)

    print(f" 折 {fold_id}/{KF_SPLITS} | "
          f"MAE={mae_val:,.4f} | MSE={mse_val:,.4f} | RMSE={rmse_val:,.4f} | R²={r2_val:.4f} "
          f"| 用时 {time.time()-t_fold:.2f}s", flush=True)

cv_MAE_mean   = float(np.mean(cv_mae_list))
cv_MAE_std    = float(np.std(cv_mae_list))
cv_MSE_mean   = float(np.mean(cv_mse_list))
cv_MSE_std    = float(np.std(cv_mse_list))
cv_RMSE_mean  = float(np.mean(cv_rmse_list))
cv_RMSE_std   = float(np.std(cv_rmse_list))
cv_R2_mean    = float(np.mean(cv_r2_list))
cv_R2_std     = float(np.std(cv_r2_list))

print("\n[步骤4] 6折交叉验证 - 平均指标", flush=True)
print(f" CV MAE   : {cv_MAE_mean:,.4f} (std {cv_MAE_std:,.4f})", flush=True)
print(f" CV MSE   : {cv_MSE_mean:,.4f} (std {cv_MSE_std:,.4f})", flush=True)
print(f" CV RMSE  : {cv_RMSE_mean:,.4f} (std {cv_RMSE_std:,.4f})", flush=True)
print(f" CV R^2   : {cv_R2_mean:.4f} (std {cv_R2_std:.4f})", flush=True)

########################################
# 12. 系数表 (特征重要性)
########################################
print("\n[步骤5] Lasso系数（特征重要性）", flush=True)

# 从训练好的完整模型里拿系数
lasso_model = lasso_pipe.named_steps["lasso"]

coef_table = pd.DataFrame({
    "feature": X_ok.columns,
    "coef": lasso_model.coef_
}).sort_values(
    by="coef",
    key=lambda s: s.abs(),
    ascending=False
)

print("Top 30 绝对值最大的系数：", flush=True)
print(coef_table.head(30).to_string(index=False), flush=True)

coef_path = "lasso_coef_alpha10000_full.csv"
coef_table.to_csv(coef_path, index=False)
print("系数表已保存 ->", coef_path, flush=True)

########################################
# 13. 最后总结（直接可以粘到报告正文）
########################################
print("\n[步骤6] 最终总结 (抄表用)", flush=True)

print("训练集(in-sample):", flush=True)
print(f" MAE={train_MAE:,.4f} | MSE={train_MSE:,.4f} | RMSE={train_RMSE:,.4f} | R²={train_R2:.4f}", flush=True)

print("测试集(out-of-sample):", flush=True)
print(f" MAE={test_MAE:,.4f} | MSE={test_MSE:,.4f} | RMSE={test_RMSE:,.4f} | R²={test_R2:.4f}", flush=True)

print("6折交叉验证(全量去极值后样本):", flush=True)
print(f" MAE均值={cv_MAE_mean:,.4f} (std {cv_MAE_std:,.4f}) | "
      f"MSE均值={cv_MSE_mean:,.4f} (std {cv_MSE_std:,.4f}) | "
      f"RMSE均值={cv_RMSE_mean:,.4f} (std {cv_RMSE_std:,.4f}) | "
      f"R²均值={cv_R2_mean:.4f} (std {cv_R2_std:.4f})", flush=True)

print("\n✅ Lasso 全量评估完成。", flush=True)


【Lasso最终版全量评估】alpha = 1000.0
数据维度: (98899, 47)
最终可用样本量: 93365 特征数: 140
训练集: (74692, 140) (74692,)
测试集: (18673, 140) (18673,)

[步骤1] 拟合训练集...
拟合完成，用时 21.22 秒

[步骤2] 训练集/测试集指标...
【训练集 / in-sample】
 MAE  : 109,795.3770
 MSE  : 23,237,738,290.5971
 RMSE : 152,439.2938
 R^2  : 0.7465

【测试集 / out-of-sample】
 MAE  : 110,068.8755
 MSE  : 23,271,469,763.3007
 RMSE : 152,549.8927
 R^2  : 0.7492

[步骤3] 6折交叉验证 (全量X_ok,y_ok)...
 折 1/6 | MAE=110,318.2165 | MSE=23,460,887,020.4991 | RMSE=153,169.4716 | R²=0.7485 | 用时 21.14s
 折 2/6 | MAE=110,039.8852 | MSE=23,204,766,859.9562 | RMSE=152,331.1093 | R²=0.7488 | 用时 30.25s
 折 3/6 | MAE=109,704.1170 | MSE=23,259,036,475.9446 | RMSE=152,509.1357 | R²=0.7472 | 用时 24.29s
 折 4/6 | MAE=109,619.0781 | MSE=23,177,975,613.6822 | RMSE=152,243.1464 | R²=0.7475 | 用时 23.23s
 折 5/6 | MAE=110,091.5816 | MSE=23,272,925,158.7426 | RMSE=152,554.6629 | R²=0.7460 | 用时 27.19s
 折 6/6 | MAE=110,169.2133 | MSE=23,444,629,152.0778 | RMSE=153,116.3909 | R²=0.7405 | 用时 31.20s

[步骤4

In [8]:
import numpy as np
import pandas as pd
import time

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

########################################
# 配置
########################################
csv_path       = "/home/mw/input/abc6966/副本ruc_Class25Q2_train_rent91.csv"
target_col     = "Price"

K_CLUSTERS     = 80
SUBSET_SIZE    = 2000      # 子样本大小
CV_FOLDS       = 4         # 折数，Ridge很快，4折足够快
RANDOM_SEED    = 42

ALPHAS_TO_TRY  = [0.1, 1.0, 10.0, 50.0, 100.0, 500.0, 1000.0, 5000.0, 10000.0]

print("【Ridge小样本调参】开始", flush=True)

########################################
# 1. 读数据
########################################
df = pd.read_csv(csv_path)
print("数据维度:", df.shape, flush=True)

########################################
# 2. 数值列处理
########################################
num_cols_candidate = [
    "Price",
    "lon","lat",
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "租赁方式_整租",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带"
]
num_cols = [c for c in num_cols_candidate if c in df.columns]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if "电梯_有" in df.columns:
    df["电梯_有"] = df["电梯_有"].fillna(0)

########################################
# 3. market_cluster80 聚类 (KMeans on each house)
########################################
cluster_basis_cols = []
for c in [
    "lon","lat",
    "房龄","绿化率","容积率","物业费",
    "房屋总数","楼栋总数",
    "Price"
]:
    if c in df.columns:
        cluster_basis_cols.append(c)

cluster_data = df[cluster_basis_cols].copy()
for c in cluster_data.columns:
    cluster_data[c] = pd.to_numeric(cluster_data[c], errors="coerce")
cluster_data = cluster_data.fillna(cluster_data.median(numeric_only=True))

scaler_cluster = StandardScaler()
cluster_scaled = scaler_cluster.fit_transform(cluster_data)

K = min(K_CLUSTERS, len(df))
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["market_cluster80"] = kmeans.fit_predict(cluster_scaled).astype(int)

########################################
# 4. 交互项
########################################
if ("面积" in df.columns) and ("房龄" in df.columns):
    df["交互_面积x房龄"] = pd.to_numeric(df["面积"], errors="coerce") * \
                         pd.to_numeric(df["房龄"], errors="coerce")
if ("面积" in df.columns) and ("楼层_高层" in df.columns):
    df["交互_面积x高层"] = pd.to_numeric(df["面积"], errors="coerce") * \
                        pd.to_numeric(df["楼层_高层"], errors="coerce")
if ("楼层_高层" in df.columns) and ("电梯_有" in df.columns):
    df["交互_高层x电梯有"] = pd.to_numeric(df["楼层_高层"], errors="coerce") * \
                          pd.to_numeric(df["电梯_有"], errors="coerce")

########################################
# 5. 组 X_full (和 Lasso时保持一致)
########################################
# 城市 dummy
if "城市" in df.columns:
    city_dum = pd.get_dummies(df["城市"], prefix="city", drop_first=False).astype(int)
else:
    city_dum = pd.DataFrame(index=df.index)

# cluster80 dummy
cluster_dum = pd.get_dummies(
    df["market_cluster80"],
    prefix="cluster80",
    drop_first=False
).astype(int)

base_feature_cols = [
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "lon","lat",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "租赁方式_整租",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带",
    "交互_面积x房龄","交互_面积x高层","交互_高层x电梯有",
]
base_feature_cols = [c for c in base_feature_cols if c in df.columns]

X_base = df[base_feature_cols].copy()
for c in X_base.columns:
    X_base[c] = pd.to_numeric(X_base[c], errors="coerce")
X_base = X_base.fillna(X_base.median(numeric_only=True))

X_full = pd.concat(
    [X_base.reset_index(drop=True),
     city_dum.reset_index(drop=True),
     cluster_dum.reset_index(drop=True)],
    axis=1
)

# 去掉常数列
X_full = X_full.loc[:, X_full.std(axis=0) > 0]

y_all = pd.to_numeric(df[target_col], errors="coerce")
y_all = y_all.fillna(y_all.median())

########################################
# 6. 用 IQR 去极值
########################################
Q1 = y_all.quantile(0.25)
Q3 = y_all.quantile(0.75)
IQR = Q3 - Q1
lower_cut = Q1 - 1.5 * IQR
upper_cut = Q3 + 1.5 * IQR
mask_ok = (y_all >= lower_cut) & (y_all <= upper_cut)

X_ok = X_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_all.loc[mask_ok].reset_index(drop=True)

print("可用样本量(调参用):", X_ok.shape[0], "特征数:", X_ok.shape[1], flush=True)

########################################
# 7. 抽子样本
########################################
n_total = len(X_ok)
n_take  = min(SUBSET_SIZE, n_total)
rng = np.random.RandomState(RANDOM_SEED)
idx = rng.choice(n_total, size=n_take, replace=False)

X_sub = X_ok.iloc[idx].copy()
y_sub = y_ok.iloc[idx].copy()

print(f"子样本: {len(X_sub)} / {n_total}", flush=True)

########################################
# 8. 扫 alpha (Ridge)
########################################
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=111)

alpha_records = []

for a in ALPHAS_TO_TRY:
    print(f"\n=== alpha = {a} (Ridge) ===", flush=True)
    t_a = time.time()

    fold_mae_list  = []
    fold_rmse_list = []
    fold_r2_list   = []

    fold_id = 0
    for tr_idx, va_idx in kf.split(X_sub):
        fold_id += 1
        X_tr = X_sub.iloc[tr_idx]
        X_va = X_sub.iloc[va_idx]
        y_tr = y_sub.iloc[tr_idx]
        y_va = y_sub.iloc[va_idx]

        pipe = make_pipeline(
            StandardScaler(with_mean=False),
            Ridge(
                alpha=a,
                fit_intercept=True,
                random_state=42
            )
        )

        t_fold = time.time()
        pipe.fit(X_tr, y_tr)
        y_hat = pipe.predict(X_va)

        mae_val  = mean_absolute_error(y_va, y_hat)
        mse_val  = mean_squared_error(y_va, y_hat)
        rmse_val = np.sqrt(mse_val)
        r2_val   = r2_score(y_va, y_hat)

        fold_mae_list.append(mae_val)
        fold_rmse_list.append(rmse_val)
        fold_r2_list.append(r2_val)

        print(f"  折 {fold_id}/{CV_FOLDS} | "
              f"MAE={mae_val:,.4f} | RMSE={rmse_val:,.4f} | R²={r2_val:.4f} "
              f"| 用时 {time.time()-t_fold:.2f}s", flush=True)

    mae_mean   = float(np.mean(fold_mae_list))
    mae_std    = float(np.std(fold_mae_list))
    rmse_mean  = float(np.mean(fold_rmse_list))
    rmse_std   = float(np.std(fold_rmse_list))
    r2_mean    = float(np.mean(fold_r2_list))
    r2_std     = float(np.std(fold_r2_list))

    elapsed = time.time() - t_a

    alpha_records.append({
        "alpha": a,
        "cv_mae_mean": mae_mean,
        "cv_mae_std":  mae_std,
        "cv_rmse_mean": rmse_mean,
        "cv_rmse_std":  rmse_std,
        "cv_r2_mean":  r2_mean,
        "cv_r2_std":   r2_std,
        "time_sec": elapsed
    })

    print(f"=== alpha = {a} 完成 ===", flush=True)
    print(f"    -> CV MAE均值={mae_mean:,.4f} ± {mae_std:,.4f}", flush=True)
    print(f"    -> CV RMSE均值={rmse_mean:,.4f} ± {rmse_std:,.4f}", flush=True)
    print(f"    -> CV R²均值={r2_mean:.4f} ± {r2_std:.4f}", flush=True)
    print(f"    -> 耗时 {elapsed:.2f}s", flush=True)

########################################
# 9. 找最优alpha (按 MAE 最小)
########################################
results_ridge = pd.DataFrame(alpha_records)
results_sorted_ridge = results_ridge.sort_values("cv_mae_mean").reset_index(drop=True)

best_alpha_ridge = results_sorted_ridge.iloc[0]["alpha"]

print("\n========== Ridge小样本调参结果 ==========")
print(results_sorted_ridge.to_string(index=False))
print("========================================")
print(f"\n>>> 建议的 Ridge alpha = {best_alpha_ridge}\n", flush=True)


【Ridge小样本调参】开始
数据维度: (98899, 47)
可用样本量(调参用): 93365 特征数: 140
子样本: 2000 / 93365

=== alpha = 0.1 (Ridge) ===
  折 1/4 | MAE=114,913.7658 | RMSE=161,499.6039 | R²=0.7329 | 用时 0.22s
  折 2/4 | MAE=113,291.2078 | RMSE=152,684.6862 | R²=0.7480 | 用时 0.20s
  折 3/4 | MAE=117,041.7352 | RMSE=161,503.6545 | R²=0.7132 | 用时 0.30s
  折 4/4 | MAE=108,013.4166 | RMSE=148,100.9779 | R²=0.7501 | 用时 0.20s
=== alpha = 0.1 完成 ===
    -> CV MAE均值=113,315.0314 ± 3,337.3627
    -> CV RMSE均值=155,947.2306 ± 5,785.9868
    -> CV R²均值=0.7360 ± 0.0147
    -> 耗时 1.24s

=== alpha = 1.0 (Ridge) ===
  折 1/4 | MAE=115,669.1111 | RMSE=162,738.5439 | R²=0.7288 | 用时 0.11s
  折 2/4 | MAE=113,098.5348 | RMSE=152,944.7810 | R²=0.7471 | 用时 0.20s
  折 3/4 | MAE=117,097.1895 | RMSE=161,793.5737 | R²=0.7122 | 用时 0.30s
  折 4/4 | MAE=108,374.7979 | RMSE=149,199.6323 | R²=0.7463 | 用时 0.30s
=== alpha = 1.0 完成 ===
    -> CV MAE均值=113,559.9083 ± 3,318.8601
    -> CV RMSE均值=156,669.1327 ± 5,761.1171
    -> CV R²均值=0.7336 ± 0.0144
    -> 耗时 

In [9]:
import numpy as np
import pandas as pd
import time

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

########################################
# 配置
########################################
csv_path        = "/home/mw/input/abc6966/副本ruc_Class25Q2_train_rent91.csv"
target_col      = "Price"
K_CLUSTERS      = 80
TEST_RATIO      = 0.2
RANDOM_SEED     = 111
KF_SPLITS       = 6

BEST_ALPHA_RIDGE = 100.0   # ← 你的小样本调参结果

print("【Ridge最终版全量评估】alpha =", BEST_ALPHA_RIDGE, flush=True)

########################################
# 1. 读数据
########################################
df = pd.read_csv(csv_path)
print("数据维度:", df.shape, flush=True)

########################################
# 2. 数值化 + 缺失处理
########################################
num_cols_candidate = [
    "Price",
    "lon","lat",
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "租赁方式_整租",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带"
]
num_cols = [c for c in num_cols_candidate if c in df.columns]

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 电梯缺就当0（无电梯）
if "电梯_有" in df.columns:
    df["电梯_有"] = df["电梯_有"].fillna(0)

########################################
# 3. 市场片区聚类：KMeans(K=80)
#    用位置+楼盘特征+价格对每套房分市场片区
########################################
cluster_basis_cols = []
for c in [
    "lon","lat",
    "房龄","绿化率","容积率","物业费",
    "房屋总数","楼栋总数",
    "Price"
]:
    if c in df.columns:
        cluster_basis_cols.append(c)

cluster_data = df[cluster_basis_cols].copy()
for c in cluster_data.columns:
    cluster_data[c] = pd.to_numeric(cluster_data[c], errors="coerce")
cluster_data = cluster_data.fillna(cluster_data.median(numeric_only=True))

scaler_cluster = StandardScaler()
cluster_scaled = scaler_cluster.fit_transform(cluster_data)

K = min(K_CLUSTERS, len(df))
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["market_cluster80"] = kmeans.fit_predict(cluster_scaled).astype(int)

########################################
# 4. 交互项（线性里允许斜率改变）
########################################
if ("面积" in df.columns) and ("房龄" in df.columns):
    df["交互_面积x房龄"] = pd.to_numeric(df["面积"], errors="coerce") * \
                         pd.to_numeric(df["房龄"], errors="coerce")

if ("面积" in df.columns) and ("楼层_高层" in df.columns):
    df["交互_面积x高层"] = pd.to_numeric(df["面积"], errors="coerce") * \
                        pd.to_numeric(df["楼层_高层"], errors="coerce")

if ("楼层_高层" in df.columns) and ("电梯_有" in df.columns):
    df["交互_高层x电梯有"] = pd.to_numeric(df["楼层_高层"], errors="coerce") * \
                          pd.to_numeric(df["电梯_有"], errors="coerce")

########################################
# 5. 构造特征矩阵 X_full
########################################
# 城市 dummy
if "城市" in df.columns:
    city_dum = pd.get_dummies(df["城市"], prefix="city", drop_first=False).astype(int)
else:
    city_dum = pd.DataFrame(index=df.index)

# cluster80 dummy
cluster_dum = pd.get_dummies(
    df["market_cluster80"],
    prefix="cluster80",
    drop_first=False
).astype(int)

# 主体特征 + 时间 + 设施 + 交互项
base_feature_cols = [
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "lon","lat",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "租赁方式_整租",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带",
    "交互_面积x房龄","交互_面积x高层","交互_高层x电梯有",
]
base_feature_cols = [c for c in base_feature_cols if c in df.columns]

X_base = df[base_feature_cols].copy()
for c in X_base.columns:
    X_base[c] = pd.to_numeric(X_base[c], errors="coerce")
X_base = X_base.fillna(X_base.median(numeric_only=True))

X_full = pd.concat(
    [X_base.reset_index(drop=True),
     city_dum.reset_index(drop=True),
     cluster_dum.reset_index(drop=True)],
    axis=1
)

# 去掉常数列
X_full = X_full.loc[:, X_full.std(axis=0) > 0]

# 目标 y
y_all = pd.to_numeric(df[target_col], errors="coerce")
y_all = y_all.fillna(y_all.median())

########################################
# 6. IQR 剔除极端值
########################################
Q1 = y_all.quantile(0.25)
Q3 = y_all.quantile(0.75)
IQR = Q3 - Q1
lower_cut = Q1 - 1.5 * IQR
upper_cut = Q3 + 1.5 * IQR
mask_ok = (y_all >= lower_cut) & (y_all <= upper_cut)

X_ok = X_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_all.loc[mask_ok].reset_index(drop=True)

print("可用样本量(最终):", X_ok.shape[0], "特征数:", X_ok.shape[1], flush=True)

########################################
# 7. 训练/测试 划分
########################################
X_train, X_test, y_train, y_test = train_test_split(
    X_ok, y_ok,
    test_size=TEST_RATIO,
    random_state=RANDOM_SEED
)

print("训练集:", X_train.shape, y_train.shape, flush=True)
print("测试集:",  X_test.shape,  y_test.shape,  flush=True)

########################################
# 8. 定义 Ridge 模型
########################################
ridge_pipe = make_pipeline(
    StandardScaler(with_mean=False),
    Ridge(
        alpha=BEST_ALPHA_RIDGE,
        fit_intercept=True,
        random_state=42
    )
)

########################################
# 9. 拟合训练集
########################################
print("\n[步骤1] 拟合训练集...", flush=True)
t0 = time.time()
ridge_pipe.fit(X_train, y_train)
print(f"拟合完成，用时 {time.time()-t0:.2f} 秒", flush=True)

########################################
# 10. 训练/测试集指标
########################################
y_pred_tr = ridge_pipe.predict(X_train)
train_MAE  = mean_absolute_error(y_train, y_pred_tr)
train_MSE  = mean_squared_error(y_train, y_pred_tr)
train_RMSE = np.sqrt(train_MSE)
train_R2   = r2_score(y_train, y_pred_tr)

y_pred_te = ridge_pipe.predict(X_test)
test_MAE  = mean_absolute_error(y_test, y_pred_te)
test_MSE  = mean_squared_error(y_test, y_pred_te)
test_RMSE = np.sqrt(test_MSE)
test_R2   = r2_score(y_test, y_pred_te)

print("【训练集 / in-sample】", flush=True)
print(f" MAE  : {train_MAE:,.4f}", flush=True)
print(f" MSE  : {train_MSE:,.4f}", flush=True)
print(f" RMSE : {train_RMSE:,.4f}", flush=True)
print(f" R^2  : {train_R2:.4f}", flush=True)

print("\n【测试集 / out-of-sample】", flush=True)
print(f" MAE  : {test_MAE:,.4f}", flush=True)
print(f" MSE  : {test_MSE:,.4f}", flush=True)
print(f" RMSE : {test_RMSE:,.4f}", flush=True)
print(f" R^2  : {test_R2:.4f}", flush=True)

########################################
# 11. 6折交叉验证 (全量 X_ok,y_ok)
########################################
print("\n[步骤2] 6折交叉验证...", flush=True)

kf = KFold(n_splits=KF_SPLITS, shuffle=True, random_state=RANDOM_SEED)

cv_mae_list  = []
cv_mse_list  = []
cv_rmse_list = []
cv_r2_list   = []

fold_id = 0
for tr_idx, va_idx in kf.split(X_ok):
    fold_id += 1

    X_tr = X_ok.iloc[tr_idx]
    X_va = X_ok.iloc[va_idx]
    y_tr = y_ok.iloc[tr_idx]
    y_va = y_ok.iloc[va_idx]

    pipe_cv = make_pipeline(
        StandardScaler(with_mean=False),
        Ridge(
            alpha=BEST_ALPHA_RIDGE,
            fit_intercept=True,
            random_state=42
        )
    )

    t_fold = time.time()
    pipe_cv.fit(X_tr, y_tr)
    y_hat = pipe_cv.predict(X_va)

    mae_val  = mean_absolute_error(y_va, y_hat)
    mse_val  = mean_squared_error(y_va, y_hat)
    rmse_val = np.sqrt(mse_val)
    r2_val   = r2_score(y_va, y_hat)

    cv_mae_list.append(mae_val)
    cv_mse_list.append(mse_val)
    cv_rmse_list.append(rmse_val)
    cv_r2_list.append(r2_val)

    print(f" 折 {fold_id}/{KF_SPLITS} | "
          f"MAE={mae_val:,.4f} | MSE={mse_val:,.4f} | RMSE={rmse_val:,.4f} | R²={r2_val:.4f} "
          f"| 用时 {time.time()-t_fold:.2f}s", flush=True)

cv_MAE_mean   = float(np.mean(cv_mae_list))
cv_MAE_std    = float(np.std(cv_mae_list))
cv_MSE_mean   = float(np.mean(cv_mse_list))
cv_MSE_std    = float(np.std(cv_mse_list))
cv_RMSE_mean  = float(np.mean(cv_rmse_list))
cv_RMSE_std   = float(np.std(cv_rmse_list))
cv_R2_mean    = float(np.mean(cv_r2_list))
cv_R2_std     = float(np.std(cv_r2_list))

print("\n[步骤3] 6折交叉验证 - 平均指标", flush=True)
print(f" CV MAE   : {cv_MAE_mean:,.4f} (std {cv_MAE_std:,.4f})", flush=True)
print(f" CV MSE   : {cv_MSE_mean:,.4f} (std {cv_MSE_std:,.4f})", flush=True)
print(f" CV RMSE  : {cv_RMSE_mean:,.4f} (std {cv_RMSE_std:,.4f})", flush=True)
print(f" CV R^2   : {cv_R2_mean:.4f} (std {cv_R2_std:.4f})", flush=True)

########################################
# 12. 系数表 (Ridge不会稀疏掉系数，但可以看大小和方向)
########################################
print("\n[步骤4] Ridge 系数Top30", flush=True)

ridge_model = ridge_pipe.named_steps["ridge"]

coef_table_ridge = pd.DataFrame({
    "feature": X_ok.columns,
    "coef": ridge_model.coef_
}).sort_values(
    by="coef",
    key=lambda s: s.abs(),
    ascending=False
)

print(coef_table_ridge.head(30).to_string(index=False), flush=True)

coef_out = "ridge_coef_alpha100_full.csv"
coef_table_ridge.to_csv(coef_out, index=False)
print("系数表已保存 ->", coef_out, flush=True)

########################################
# 13. 总结 (可以直接放到报告里)
########################################
print("\n[步骤5] 最终总结 (抄表用)", flush=True)

print("训练集(in-sample):", flush=True)
print(f" MAE={train_MAE:,.4f} | MSE={train_MSE:,.4f} | RMSE={train_RMSE:,.4f} | R²={train_R2:.4f}", flush=True)

print("测试集(out-of-sample):", flush=True)
print(f" MAE={test_MAE:,.4f} | MSE={test_MSE:,.4f} | RMSE={test_RMSE:,.4f} | R²={test_R2:.4f}", flush=True)

print("6折交叉验证(全量):", flush=True)
print(f" MAE均值={cv_MAE_mean:,.4f} (std {cv_MAE_std:,.4f}) | "
      f"MSE均值={cv_MSE_mean:,.4f} (std {cv_MSE_std:,.4f}) | "
      f"RMSE均值={cv_RMSE_mean:,.4f} (std {cv_RMSE_std:,.4f}) | "
      f"R²均值={cv_R2_mean:.4f} (std {cv_R2_std:.4f})", flush=True)

print("\n✅ Ridge 全量评估完成。", flush=True)


【Ridge最终版全量评估】alpha = 100.0
数据维度: (98899, 47)
可用样本量(最终): 93365 特征数: 140
训练集: (74692, 140) (74692,)
测试集: (18673, 140) (18673,)

[步骤1] 拟合训练集...
拟合完成，用时 0.58 秒
【训练集 / in-sample】
 MAE  : 109,126.6149
 MSE  : 22,832,773,801.9475
 RMSE : 151,105.1746
 R^2  : 0.7510

【测试集 / out-of-sample】
 MAE  : 109,431.2950
 MSE  : 22,849,940,020.9413
 RMSE : 151,161.9662
 R^2  : 0.7538

[步骤2] 6折交叉验证...
 折 1/6 | MAE=109,608.9221 | MSE=22,998,125,423.0593 | RMSE=151,651.3285 | R²=0.7535 | 用时 0.72s
 折 2/6 | MAE=109,439.9347 | MSE=22,874,753,533.9720 | RMSE=151,244.0198 | R²=0.7524 | 用时 0.76s
 折 3/6 | MAE=109,020.2221 | MSE=22,841,894,211.0044 | RMSE=151,135.3506 | R²=0.7517 | 用时 0.75s
 折 4/6 | MAE=109,029.1092 | MSE=22,790,164,965.0527 | RMSE=150,964.1181 | R²=0.7517 | 用时 0.73s
 折 5/6 | MAE=109,250.2383 | MSE=22,792,722,470.6548 | RMSE=150,972.5885 | R²=0.7512 | 用时 0.74s
 折 6/6 | MAE=109,637.8626 | MSE=23,094,813,102.6816 | RMSE=151,969.7769 | R²=0.7444 | 用时 0.74s

[步骤3] 6折交叉验证 - 平均指标
 CV MAE   : 109,331.0482

In [12]:
import numpy as np
import pandas as pd
import time

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

########################################
# 基础配置
########################################
csv_path       = "/home/mw/input/abc6966/副本ruc_Class25Q2_train_rent91.csv"
target_col     = "Price"

K_CLUSTERS     = 80
SUBSET_SIZE    = 2000      # 抽子样本大小
CV_FOLDS       = 4         # 用4折，让速度可接受
RANDOM_SEED    = 42

# 要扫描的网格：
ALPHAS_TO_TRY     = [10.0, 50.0, 100.0, 500.0, 1000.0]   # 和 Lasso/Ridge同量级
L1_RATIOS_TO_TRY  = [0.2, 0.3,0.4,0.5, 0.8,0.9,0.95]                      # 0.2更像Ridge, 0.8更像Lasso

print("【Elastic Net 小样本调参】开始", flush=True)

########################################
# 1. 读数据
########################################
df = pd.read_csv(csv_path)
print("数据维度:", df.shape, flush=True)

########################################
# 2. 数值列 & 缺失
########################################
num_cols_candidate = [
    "Price",
    "lon","lat",
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "租赁方式_整租",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带"
]
num_cols = [c for c in num_cols_candidate if c in df.columns]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if "电梯_有" in df.columns:
    df["电梯_有"] = df["电梯_有"].fillna(0)

########################################
# 3. KMeans 聚类成 80 个市场片区
########################################
cluster_basis_cols = []
for c in [
    "lon","lat",
    "房龄","绿化率","容积率","物业费",
    "房屋总数","楼栋总数",
    "Price"
]:
    if c in df.columns:
        cluster_basis_cols.append(c)

cluster_data = df[cluster_basis_cols].copy()
for c in cluster_data.columns:
    cluster_data[c] = pd.to_numeric(cluster_data[c], errors="coerce")
cluster_data = cluster_data.fillna(cluster_data.median(numeric_only=True))

scaler_cluster = StandardScaler()
cluster_scaled = scaler_cluster.fit_transform(cluster_data)

K = min(K_CLUSTERS, len(df))
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["market_cluster80"] = kmeans.fit_predict(cluster_scaled).astype(int)

########################################
# 4. 交互项
########################################
if ("面积" in df.columns) and ("房龄" in df.columns):
    df["交互_面积x房龄"] = pd.to_numeric(df["面积"], errors="coerce") * \
                         pd.to_numeric(df["房龄"], errors="coerce")

if ("面积" in df.columns) and ("楼层_高层" in df.columns):
    df["交互_面积x高层"] = pd.to_numeric(df["面积"], errors="coerce") * \
                        pd.to_numeric(df["楼层_高层"], errors="coerce")

if ("楼层_高层" in df.columns) and ("电梯_有" in df.columns):
    df["交互_高层x电梯有"] = pd.to_numeric(df["楼层_高层"], errors="coerce") * \
                          pd.to_numeric(df["电梯_有"], errors="coerce")

########################################
# 5. 组特征矩阵 X_full（和 Ridge/Lasso 保持一致）
########################################
# 城市 dummy
if "城市" in df.columns:
    city_dum = pd.get_dummies(df["城市"], prefix="city", drop_first=False).astype(int)
else:
    city_dum = pd.DataFrame(index=df.index)

# cluster80 dummy
cluster_dum = pd.get_dummies(
    df["market_cluster80"],
    prefix="cluster80",
    drop_first=False
).astype(int)

base_feature_cols = [
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "lon","lat",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "租赁方式_整租",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带",
    "交互_面积x房龄","交互_面积x高层","交互_高层x电梯有",
]
base_feature_cols = [c for c in base_feature_cols if c in df.columns]

X_base = df[base_feature_cols].copy()
for c in X_base.columns:
    X_base[c] = pd.to_numeric(X_base[c], errors="coerce")
X_base = X_base.fillna(X_base.median(numeric_only=True))

X_full = pd.concat(
    [X_base.reset_index(drop=True),
     city_dum.reset_index(drop=True),
     cluster_dum.reset_index(drop=True)],
    axis=1
)

# 去掉常数列
X_full = X_full.loc[:, X_full.std(axis=0) > 0]

# 目标变量
y_all = pd.to_numeric(df[target_col], errors="coerce")
y_all = y_all.fillna(y_all.median())

########################################
# 6. IQR 去极值，得到 X_ok / y_ok
########################################
Q1 = y_all.quantile(0.25)
Q3 = y_all.quantile(0.75)
IQR = Q3 - Q1
lower_cut = Q1 - 1.5 * IQR
upper_cut = Q3 + 1.5 * IQR
mask_ok = (y_all >= lower_cut) & (y_all <= upper_cut)

X_ok = X_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_all.loc[mask_ok].reset_index(drop=True)

print("可用样本量(调参):", X_ok.shape[0], "特征数:", X_ok.shape[1], flush=True)

########################################
# 7. 抽子样本
########################################
n_total = len(X_ok)
n_take  = min(SUBSET_SIZE, n_total)
rng = np.random.RandomState(RANDOM_SEED)
idx = rng.choice(n_total, size=n_take, replace=False)

X_sub = X_ok.iloc[idx].copy()
y_sub = y_ok.iloc[idx].copy()

print(f"子样本: {len(X_sub)} / {n_total}", flush=True)

########################################
# 8. 4折CV 扫 (alpha, l1_ratio)
########################################
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=111)

results_records = []

for a in ALPHAS_TO_TRY:
    for l1r in L1_RATIOS_TO_TRY:
        combo_name = f"alpha={a}, l1={l1r}"
        print(f"\n=== {combo_name} ===", flush=True)
        t_combo = time.time()

        fold_mae_list  = []
        fold_rmse_list = []
        fold_r2_list   = []

        fold_id = 0
        for tr_idx, va_idx in kf.split(X_sub):
            fold_id += 1
            X_tr = X_sub.iloc[tr_idx]
            X_va = X_sub.iloc[va_idx]
            y_tr = y_sub.iloc[tr_idx]
            y_va = y_sub.iloc[va_idx]

            pipe = make_pipeline(
                StandardScaler(with_mean=False),
                ElasticNet(
                    alpha=a,
                    l1_ratio=l1r,
                    fit_intercept=True,
                    max_iter=10000,
                    random_state=42
                )
            )

            t_fold = time.time()
            pipe.fit(X_tr, y_tr)
            y_hat = pipe.predict(X_va)

            mae_val  = mean_absolute_error(y_va, y_hat)
            mse_val  = mean_squared_error(y_va, y_hat)
            rmse_val = np.sqrt(mse_val)
            r2_val   = r2_score(y_va, y_hat)

            fold_mae_list.append(mae_val)
            fold_rmse_list.append(rmse_val)
            fold_r2_list.append(r2_val)

            print(f"  折 {fold_id}/{CV_FOLDS} | "
                  f"MAE={mae_val:,.4f} | RMSE={rmse_val:,.4f} | R²={r2_val:.4f} "
                  f"| 用时 {time.time()-t_fold:.2f}s", flush=True)

        mae_mean   = float(np.mean(fold_mae_list))
        mae_std    = float(np.std(fold_mae_list))
        rmse_mean  = float(np.mean(fold_rmse_list))
        rmse_std   = float(np.std(fold_rmse_list))
        r2_mean    = float(np.mean(fold_r2_list))
        r2_std     = float(np.std(fold_r2_list))
        combo_time = time.time() - t_combo

        print(f"=== {combo_name} 结束 ===", flush=True)
        print(f"    -> CV MAE均值={mae_mean:,.4f} ± {mae_std:,.4f}", flush=True)
        print(f"    -> CV RMSE均值={rmse_mean:,.4f} ± {rmse_std:,.4f}", flush=True)
        print(f"    -> CV R²均值={r2_mean:.4f} ± {r2_std:.4f}", flush=True)
        print(f"    -> 耗时 {combo_time:.2f}s", flush=True)

        results_records.append({
            "alpha": a,
            "l1_ratio": l1r,
            "cv_mae_mean": mae_mean,
            "cv_mae_std": mae_std,
            "cv_rmse_mean": rmse_mean,
            "cv_rmse_std": rmse_std,
            "cv_r2_mean": r2_mean,
            "cv_r2_std": r2_std,
            "time_sec": combo_time
        })

########################################
# 9. 选最优 (按 MAE最小)
########################################
results_df = pd.DataFrame(results_records)
results_sorted = results_df.sort_values("cv_mae_mean").reset_index(drop=True)

best_alpha_en    = results_sorted.loc[0, "alpha"]
best_l1ratio_en  = results_sorted.loc[0, "l1_ratio"]

print("\n========== Elastic Net 小样本调参结果 ==========")
print(results_sorted.to_string(index=False))
print("================================================")
print(f"\n>>> 结论：推荐 alpha = {best_alpha_en}, l1_ratio = {best_l1ratio_en}\n", flush=True)


【Elastic Net 小样本调参】开始
数据维度: (98899, 47)
可用样本量(调参): 93365 特征数: 140
子样本: 2000 / 93365

=== alpha=10.0, l1=0.2 ===
  折 1/4 | MAE=209,080.3820 | RMSE=265,857.3585 | R²=0.2762 | 用时 0.06s
  折 2/4 | MAE=203,294.6319 | RMSE=257,304.5034 | R²=0.2842 | 用时 0.29s
  折 3/4 | MAE=196,357.4590 | RMSE=253,684.5555 | R²=0.2924 | 用时 0.29s
  折 4/4 | MAE=204,951.5756 | RMSE=251,971.7026 | R²=0.2765 | 用时 0.20s
=== alpha=10.0, l1=0.2 结束 ===
    -> CV MAE均值=203,421.0121 ± 4,590.2300
    -> CV RMSE均值=257,204.5300 ± 5,353.8337
    -> CV R²均值=0.2823 ± 0.0066
    -> 耗时 0.97s

=== alpha=10.0, l1=0.3 ===
  折 1/4 | MAE=205,024.0968 | RMSE=261,185.4277 | R²=0.3014 | 用时 0.11s
  折 2/4 | MAE=199,141.2849 | RMSE=252,727.2885 | R²=0.3095 | 用时 0.20s
  折 3/4 | MAE=192,112.9769 | RMSE=249,072.6521 | R²=0.3179 | 用时 0.40s
  折 4/4 | MAE=200,812.2702 | RMSE=247,571.7490 | R²=0.3016 | 用时 0.20s
=== alpha=10.0, l1=0.3 结束 ===
    -> CV MAE均值=199,272.6572 ± 4,656.3866
    -> CV RMSE均值=252,639.2793 ± 5,278.3779
    -> CV R²均值=0.3076 ±

In [13]:
import numpy as np
import pandas as pd
import time

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

########################################
# 配置：把下面两个值改成你刚刚小样本调参得到的最优组合
########################################
csv_path        = "/home/mw/input/abc6966/副本ruc_Class25Q2_train_rent91.csv"
target_col      = "Price"
K_CLUSTERS      = 80
TEST_RATIO      = 0.2
RANDOM_SEED     = 111
KF_SPLITS       = 6

BEST_ALPHA_EN    =10.0
BEST_L1R_EN      = 0.999

print("【Elastic Net 最终全量评估】alpha =", BEST_ALPHA_EN,
      " l1_ratio =", BEST_L1R_EN, flush=True)

########################################
# 1. 读数据
########################################
df = pd.read_csv(csv_path)
print("数据维度:", df.shape, flush=True)

########################################
# 2. 数值化
########################################
num_cols_candidate = [
    "Price",
    "lon","lat",
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "租赁方式_整租",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带"
]
num_cols = [c for c in num_cols_candidate if c in df.columns]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if "电梯_有" in df.columns:
    df["电梯_有"] = df["电梯_有"].fillna(0)

########################################
# 3. KMeans 聚类 -> market_cluster80
########################################
cluster_basis_cols = []
for c in [
    "lon","lat",
    "房龄","绿化率","容积率","物业费",
    "房屋总数","楼栋总数",
    "Price"
]:
    if c in df.columns:
        cluster_basis_cols.append(c)

cluster_data = df[cluster_basis_cols].copy()
for c in cluster_data.columns:
    cluster_data[c] = pd.to_numeric(cluster_data[c], errors="coerce")
cluster_data = cluster_data.fillna(cluster_data.median(numeric_only=True))

scaler_cluster = StandardScaler()
cluster_scaled = scaler_cluster.fit_transform(cluster_data)

K = min(K_CLUSTERS, len(df))
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["market_cluster80"] = kmeans.fit_predict(cluster_scaled).astype(int)

########################################
# 4. 交互项
########################################
if ("面积" in df.columns) and ("房龄" in df.columns):
    df["交互_面积x房龄"] = pd.to_numeric(df["面积"], errors="coerce") * \
                         pd.to_numeric(df["房龄"], errors="coerce")

if ("面积" in df.columns) and ("楼层_高层" in df.columns):
    df["交互_面积x高层"] = pd.to_numeric(df["面积"], errors="coerce") * \
                        pd.to_numeric(df["楼层_高层"], errors="coerce")

if ("楼层_高层" in df.columns) and ("电梯_有" in df.columns):
    df["交互_高层x电梯有"] = pd.to_numeric(df["楼层_高层"], errors="coerce") * \
                          pd.to_numeric(df["电梯_有"], errors="coerce")

########################################
# 5. 构造 X_full
########################################
# 城市 dummy
if "城市" in df.columns:
    city_dum = pd.get_dummies(df["城市"], prefix="city", drop_first=False).astype(int)
else:
    city_dum = pd.DataFrame(index=df.index)

# cluster80 dummy
cluster_dum = pd.get_dummies(
    df["market_cluster80"],
    prefix="cluster80",
    drop_first=False
).astype(int)

base_feature_cols = [
    "面积","室","房屋总数","楼栋总数","绿化率","容积率","物业费",
    "房龄",
    "lon","lat",
    "楼层_中层","楼层_高层","楼层_地下室",
    "电梯_有",
    "付款方式_双月付价","付款方式_季付价","付款方式_年付价","付款方式_月付价",
    "租赁方式_整租",
    "朝向_东","朝向_南","朝向_西","朝向_北",
    "年_2025",
    "月_1","月_2","月_3","月_4","月_6","月_7","月_8","月_9","月_10","月_11","月_12",
    "设施_床","设施_衣柜","设施_空调","设施_洗衣机","设施_热水器",
    "设施_冰箱","设施_天然气","设施_电视","设施_暖气","设施_宽带",
    "交互_面积x房龄","交互_面积x高层","交互_高层x电梯有",
]
base_feature_cols = [c for c in base_feature_cols if c in df.columns]

X_base = df[base_feature_cols].copy()
for c in X_base.columns:
    X_base[c] = pd.to_numeric(X_base[c], errors="coerce")
X_base = X_base.fillna(X_base.median(numeric_only=True))

X_full = pd.concat(
    [X_base.reset_index(drop=True),
     city_dum.reset_index(drop=True),
     cluster_dum.reset_index(drop=True)],
    axis=1
)

# 去掉常数列
X_full = X_full.loc[:, X_full.std(axis=0) > 0]

# y
y_all = pd.to_numeric(df[target_col], errors="coerce")
y_all = y_all.fillna(y_all.median())

########################################
# 6. IQR 去极值
########################################
Q1 = y_all.quantile(0.25)
Q3 = y_all.quantile(0.75)
IQR = Q3 - Q1
lower_cut = Q1 - 1.5 * IQR
upper_cut = Q3 + 1.5 * IQR
mask_ok = (y_all >= lower_cut) & (y_all <= upper_cut)

X_ok = X_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_all.loc[mask_ok].reset_index(drop=True)

print("最终可用样本量:", X_ok.shape[0], "特征数:", X_ok.shape[1], flush=True)

########################################
# 7. train / test 切分
########################################
X_train, X_test, y_train, y_test = train_test_split(
    X_ok, y_ok,
    test_size=TEST_RATIO,
    random_state=RANDOM_SEED
)

print("训练集:", X_train.shape, y_train.shape, flush=True)
print("测试集:",  X_test.shape,  y_test.shape,  flush=True)

########################################
# 8. 建最终弹性网络模型
########################################
enet_pipe = make_pipeline(
    StandardScaler(with_mean=False),
    ElasticNet(
        alpha=BEST_ALPHA_EN,
        l1_ratio=BEST_L1R_EN,
        fit_intercept=True,
        max_iter=10000,
        random_state=42
    )
)

########################################
# 9. 拟合训练集
########################################
print("\n[步骤1] 拟合训练集...", flush=True)
t0 = time.time()
enet_pipe.fit(X_train, y_train)
print(f"拟合完成，用时 {time.time()-t0:.2f} 秒", flush=True)

########################################
# 10. 训练/测试指标
########################################
y_pred_tr = enet_pipe.predict(X_train)
train_MAE  = mean_absolute_error(y_train, y_pred_tr)
train_MSE  = mean_squared_error(y_train, y_pred_tr)
train_RMSE = np.sqrt(train_MSE)
train_R2   = r2_score(y_train, y_pred_tr)

y_pred_te = enet_pipe.predict(X_test)
test_MAE  = mean_absolute_error(y_test, y_pred_te)
test_MSE  = mean_squared_error(y_test, y_pred_te)
test_RMSE = np.sqrt(test_MSE)
test_R2   = r2_score(y_test, y_pred_te)

print("【训练集 / in-sample】", flush=True)
print(f" MAE  : {train_MAE:,.4f}", flush=True)
print(f" MSE  : {train_MSE:,.4f}", flush=True)
print(f" RMSE : {train_RMSE:,.4f}", flush=True)
print(f" R^2  : {train_R2:.4f}", flush=True)

print("\n【测试集 / out-of-sample】", flush=True)
print(f" MAE  : {test_MAE:,.4f}", flush=True)
print(f" MSE  : {test_MSE:,.4f}", flush=True)
print(f" RMSE : {test_RMSE:,.4f}", flush=True)
print(f" R^2  : {test_R2:.4f}", flush=True)

########################################
# 11. 6折CV（全量 X_ok, y_ok）
########################################
print("\n[步骤2] 6折交叉验证 (全量X_ok,y_ok)...", flush=True)

kf = KFold(n_splits=KF_SPLITS, shuffle=True, random_state=RANDOM_SEED)

cv_mae_list  = []
cv_mse_list  = []
cv_rmse_list = []
cv_r2_list   = []

fold_id = 0
for tr_idx, va_idx in kf.split(X_ok):
    fold_id += 1

    X_tr = X_ok.iloc[tr_idx]
    X_va = X_ok.iloc[va_idx]
    y_tr = y_ok.iloc[tr_idx]
    y_va = y_ok.iloc[va_idx]

    pipe_cv = make_pipeline(
        StandardScaler(with_mean=False),
        ElasticNet(
            alpha=BEST_ALPHA_EN,
            l1_ratio=BEST_L1R_EN,
            fit_intercept=True,
            max_iter=10000,
            random_state=42
        )
    )

    t_fold = time.time()
    pipe_cv.fit(X_tr, y_tr)
    y_hat = pipe_cv.predict(X_va)

    mae_val  = mean_absolute_error(y_va, y_hat)
    mse_val  = mean_squared_error(y_va, y_hat)
    rmse_val = np.sqrt(mse_val)
    r2_val   = r2_score(y_va, y_hat)

    cv_mae_list.append(mae_val)
    cv_mse_list.append(mse_val)
    cv_rmse_list.append(rmse_val)
    cv_r2_list.append(r2_val)

    print(f" 折 {fold_id}/{KF_SPLITS} | "
          f"MAE={mae_val:,.4f} | MSE={mse_val:,.4f} | RMSE={rmse_val:,.4f} | R²={r2_val:.4f} "
          f"| 用时 {time.time()-t_fold:.2f}s", flush=True)

cv_MAE_mean   = float(np.mean(cv_mae_list))
cv_MAE_std    = float(np.std(cv_mae_list))
cv_MSE_mean   = float(np.mean(cv_mse_list))
cv_MSE_std    = float(np.std(cv_mse_list))
cv_RMSE_mean  = float(np.mean(cv_rmse_list))
cv_RMSE_std   = float(np.std(cv_rmse_list))
cv_R2_mean    = float(np.mean(cv_r2_list))
cv_R2_std     = float(np.std(cv_r2_list))

print("\n[步骤3] 6折交叉验证 - 平均指标", flush=True)
print(f" CV MAE   : {cv_MAE_mean:,.4f} (std {cv_MAE_std:,.4f})", flush=True)
print(f" CV MSE   : {cv_MSE_mean:,.4f} (std {cv_MSE_std:,.4f})", flush=True)
print(f" CV RMSE  : {cv_RMSE_mean:,.4f} (std {cv_RMSE_std:,.4f})", flush=True)
print(f" CV R^2   : {cv_R2_mean:.4f} (std {cv_R2_std:.4f})", flush=True)

########################################
# 12. 系数Top30
########################################
print("\n[步骤4] 弹性网络系数Top30", flush=True)

enet_model = enet_pipe.named_steps["elasticnet"]

coef_table_en = pd.DataFrame({
    "feature": X_ok.columns,
    "coef": enet_model.coef_
}).sort_values(
    by="coef",
    key=lambda s: s.abs(),
    ascending=False
)

print(coef_table_en.head(30).to_string(index=False), flush=True)

coef_out = "elasticnet_coef_full.csv"
coef_table_en.to_csv(coef_out, index=False)
print("系数表已保存 ->", coef_out, flush=True)

########################################
# 13. 总结（抄进报告表）
########################################
print("\n[步骤5] 最终总结 (抄表用)", flush=True)

print("训练集(in-sample):", flush=True)
print(f" MAE={train_MAE:,.4f} | MSE={train_MSE:,.4f} | RMSE={train_RMSE:,.4f} | R²={train_R2:.4f}", flush=True)

print("测试集(out-of-sample):", flush=True)
print(f" MAE={test_MAE:,.4f} | MSE={test_MSE:,.4f} | RMSE={test_RMSE:,.4f} | R²={test_R2:.4f}", flush=True)

print("6折交叉验证(全量):", flush=True)
print(f" MAE均值={cv_MAE_mean:,.4f} (std {cv_MAE_std:,.4f}) | "
      f"MSE均值={cv_MSE_mean:,.4f} (std {cv_MSE_std:,.4f}) | "
      f"RMSE均值={cv_RMSE_mean:,.4f} (std {cv_RMSE_std:,.4f}) | "
      f"R²均值={cv_R2_mean:.4f} (std {cv_R2_std:.4f})", flush=True)

print("\n✅ Elastic Net 全量评估完成。", flush=True)


【Elastic Net 最终全量评估】alpha = 10.0  l1_ratio = 0.999
数据维度: (98899, 47)
最终可用样本量: 93365 特征数: 140
训练集: (74692, 140) (74692,)
测试集: (18673, 140) (18673,)

[步骤1] 拟合训练集...
拟合完成，用时 67.02 秒
【训练集 / in-sample】
 MAE  : 109,466.4697
 MSE  : 22,998,796,527.2712
 RMSE : 151,653.5411
 R^2  : 0.7491

【测试集 / out-of-sample】
 MAE  : 109,782.9285
 MSE  : 23,038,669,630.7168
 RMSE : 151,784.9453
 R^2  : 0.7518

[步骤2] 6折交叉验证 (全量X_ok,y_ok)...
 折 1/6 | MAE=109,999.9381 | MSE=23,205,970,443.2403 | RMSE=152,335.0598 | R²=0.7513 | 用时 66.72s
 折 2/6 | MAE=109,740.5862 | MSE=23,011,666,666.2338 | RMSE=151,695.9679 | R²=0.7509 | 用时 74.01s
 折 3/6 | MAE=109,414.1342 | MSE=23,024,457,996.1197 | RMSE=151,738.1231 | R²=0.7497 | 用时 73.19s
 折 4/6 | MAE=109,357.7307 | MSE=22,955,771,095.2600 | RMSE=151,511.6203 | R²=0.7499 | 用时 67.17s
 折 5/6 | MAE=109,649.4149 | MSE=22,975,928,690.4846 | RMSE=151,578.1273 | R²=0.7492 | 用时 76.15s
 折 6/6 | MAE=109,864.5213 | MSE=23,242,815,345.1432 | RMSE=152,455.9456 | R²=0.7427 | 用时 65.75s

[步

In [ ]:
# 查看个人持久化工作区文件
!ls /home/mw/project/

In [ ]:
# 查看当前挂载的数据集目录
!ls /home/mw/input/